<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v3.0</br>
📅 Last Updated: 2026-03-22</br>
</div>

</br>

In [ ]:
# TODO 0: 001에서 저장한 토크나이저를 로드하고 디바이스를 설정해봅시다.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tokenizers import BertWordPieceTokenizer
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정
font_path = fm.findfont(fm.FontProperties(family=['AppleGothic', 'NanumGothic', 'Malgun Gothic']))
plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False

tokenizer = BertWordPieceTokenizer("naver_wp-vocab.txt", lowercase=False, strip_accents=False)
vocab_size = tokenizer.get_vocab_size()
print(f"어휘 크기: {vocab_size:,}")

# 디바이스 자동 감지 (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"디바이스: {device}")

</br>

# 학습 내용
>이번 장에서는 <strong>LSTM과 Attention 메커니즘(LSTM & Attention Mechanism)</strong>에 대해 학습합니다.</br></br>
>LSTM의 게이트 구조로 RNN의 장기 의존성 문제를 해결하고, Attention으로 중요한 토큰에 집중하는 방법을 배웁니다.</br></br>
>NSMC(네이버 영화 리뷰) 데이터셋으로 감성 분류 모델을 단계별로 구현하며 RNN → LSTM → LSTM+Attention 성능을 비교합니다.

<div style="text-align:center">

</div>

</br>

# LSTM (Long Short-Term Memory)
> LSTM은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">게이트 메커니즘으로 장기 의존성 문제를 해결</mark>한 RNN 변형입니다.

> RNN은 긴 문장을 기억하지 못합니다. "오늘 아침 서울에서 출발한 기차가 부산에 도착했다"에서 RNN이 "도착했다"를 처리할 때 "기차"의 정보는 이미 희미해져 있습니다.</br></br>
> 역전파 시 기울기는 시간 축을 따라 반복 곱해지면서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기하급수적으로 소실</mark>되고, 시퀀스가 길수록 초기 정보가 사라지는 기울기 소실(Vanishing Gradient) 문제가 발생합니다.</br></br></br>
> LSTM은 셀 상태(Cell State)라는 별도 통로로 이 문제를 해결합니다.</br></br>
> 시그모이드 함수(출력 0~1)를 활용한 세 개의 게이트가 정보 흐름을 정밀하게 제어하는데, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Forget Gate(잊기), Input Gate(기억), Output Gate(출력)</mark>으로 구성됩니다.

## LSTM 셀 구조

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">게이트</th>
      <th>역할</th>
      <th>수식</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Forget Gate ($f_t$)</td><td>이전 정보 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">삭제 비율</mark> 결정</td><td>$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$</td></tr>
    <tr><td style="text-align:center">Input Gate ($i_t$)</td><td>새 정보 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">저장 비율</mark> 결정</td><td>$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$</td></tr>
    <tr><td style="text-align:center">Output Gate ($o_t$)</td><td>셀 상태에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">출력 비율</mark> 결정</td><td>$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$</td></tr>
  </tbody>
</table>

<div style="text-align:center">

</div>

💡LSTM vs GRU
> GRU는 게이트가 2개(Reset, Update)로 LSTM보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">파라미터가 적고 빠릅니다</mark>.</br>
> 성능은 비슷한 경우가 많아 데이터 크기에 따라 선택합니다.

</br>

# 데이터 준비: NSMC 감성 분류
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ratings.txt</mark>는 네이버 영화 리뷰 데이터입니다.</br></br>
> 컬럼: `id`, `document`(리뷰 텍스트), `label`(0=부정, 1=긍정)</br></br>
> 002에서 학습한 RNN 분류기와 동일한 데이터로 LSTM과 LSTM+Attention 성능을 비교합니다.

In [ ]:
# TODO 1: NSMCDataset 클래스를 정의해봅시다.

class NSMCDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=64):
        self.samples = []
        with open(file_path, encoding="utf-8") as f:
            next(f)  # 헤더 건너뜀
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) != 3 or parts[1] == "" or parts[2] not in ("0", "1"):
                    continue
                # TODO 1-1: 문서를 토크나이저로 인코딩하고 max_length만큼 패딩/자르기해봅시다.
                encoded = tokenizer.encode(parts[1])
                ids = encoded.ids[:max_length]
                ids += [0] * (max_length - len(ids))  # 패딩
                self.samples.append((ids, int(parts[2])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, label = self.samples[idx]
        return torch.LongTensor(ids), torch.tensor(label)

dataset = NSMCDataset("ratings.txt", tokenizer)
print(f"전체 샘플 수: {len(dataset):,}")
ids, label = dataset[0]
print(f"첫 샘플 - 토큰 ID shape: {ids.shape}, 레이블: {label.item()} ({'긍정' if label.item()==1 else '부정'})")

In [ ]:
# TODO 2: 전체 데이터를 8:2로 나눠 DataLoader를 만들어봅시다.

from torch.utils.data import random_split

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size

# TODO 2-1: random_split으로 훈련/테스트 데이터셋을 분리해봅시다.
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"훈련 샘플: {train_size:,}  /  테스트 샘플: {test_size:,}")
print(f"훈련 배치 수: {len(train_loader)}")

</br>

# LSTM 감성 분류기
> Embedding → LSTM → 마지막 hidden → FC → 긍정/부정</br></br>
> RNN과 달리 LSTM은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">hidden_state와 cell_state를 함께</mark> 전달합니다.

<div style="text-align:center">

</div>

In [ ]:
# TODO 3: LSTMClassifier 클래스를 정의해봅시다.

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super().__init__()
        # TODO 3-1: 임베딩 레이어를 초기화해봅시다.
        # TODO 3-2: LSTM 레이어를 초기화해봅시다. (batch_first=True)
        # TODO 3-3: 드롭아웃 레이어를 초기화해봅시다. (p=0.3)
        # TODO 3-4: 선형 분류 레이어를 초기화해봅시다. (hidden_dim → num_classes)

    def forward(self, x):
        # TODO 3-5: 입력을 임베딩해봅시다.
        # TODO 3-6: LSTM에 통과시켜 마지막 hidden_state를 추출해봅시다.
        # TODO 3-7: 드롭아웃 후 FC 레이어로 분류해봅시다.
        pass

lstm_model = LSTMClassifier(vocab_size=vocab_size, embed_dim=256, hidden_dim=256).to(device)
print(f"파라미터 수: {sum(p.numel() for p in lstm_model.parameters()):,}")

💡LSTM 출력 구조
> `output`: 모든 시점의 hidden_state → shape `(batch, seq_len, hidden_dim)`</br>
> `hidden_state`: 마지막 시점의 hidden_state → shape `(num_layers, batch, hidden_dim)`</br>
> `cell_state`: 장기 기억 — RNN에는 없고 LSTM에만 있습니다.</br></br>
> 분류 작업에서는 `hidden_state[-1]`(마지막 레이어의 마지막 시점)을 주로 사용합니다.

</br>

## LSTM 학습

In [ ]:
# TODO 4: cross_entropy와 optimizer를 정의해봅시다.

cross_entropy = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

In [ ]:
# TODO 5: LSTM 모델을 5 에폭 학습시켜봅시다.

epochs = 5
lstm_train_loss_list = []
lstm_train_accuracy_list = []

for epoch in range(epochs):
    lstm_model.train()
    running_loss = 0.0
    correct_count = 0
    total_count = 0

    for token_ids, labels in train_loader:
        token_ids = token_ids.to(device)
        labels = labels.to(device)

        # TODO 5-1: 기울기를 초기화해봅시다.
        optimizer.zero_grad()                               # 이전 배치의 기울기 초기화
        # TODO 5-2: 순전파를 수행해봅시다.
        outputs = lstm_model(token_ids)                     # 순전파: 입력 → 모델 → 예측값
        # TODO 5-3: 손실을 계산해봅시다.
        loss = cross_entropy(outputs, labels)               # 예측 vs 정답 → 오차 측정
        # TODO 5-4: 역전파를 수행해봅시다.
        loss.backward()                                     # 역전파: 각 파라미터 기울기 계산
        # TODO 5-5: 파라미터를 업데이트해봅시다.
        optimizer.step()                                    # 기울기 방향으로 파라미터 업데이트

        running_loss  += loss.item() * labels.size(0)       # 배치 평균 × 샘플 수 = 배치 총 손실
        _, predicted   = outputs.max(1)
        correct_count += (predicted == labels).sum().item()
        total_count   += labels.size(0)

    epoch_loss     = running_loss / total_count
    epoch_accuracy = correct_count / total_count
    lstm_train_loss_list.append(epoch_loss)
    lstm_train_accuracy_list.append(epoch_accuracy)
    print(f"Epoch {epoch+1}/{epochs}  loss: {epoch_loss:.4f}  accuracy: {epoch_accuracy:.4f}")

</br>

## LSTM 평가

In [ ]:
# TODO 6: LSTM 모델의 테스트 정확도를 측정해봅시다.

lstm_model.eval()
correct_count = 0
total_count = 0

with torch.no_grad():
    for token_ids, labels in test_loader:
        token_ids = token_ids.to(device)
        labels    = labels.to(device)
        outputs   = lstm_model(token_ids)
        _, predicted = outputs.max(1)
        correct_count += (predicted == labels).sum().item()
        total_count   += labels.size(0)

lstm_accuracy = correct_count / total_count
print(f"LSTM 테스트 정확도: {lstm_accuracy:.4f}  ({lstm_accuracy*100:.2f}%)")
print(f"(002 RNN 결과와 비교해보세요)")

</br>

# Attention 메커니즘 (Attention Mechanism)
> 디코더가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 시퀀스의 관련 부분에 선택적으로 집중</mark>할 수 있게 하는 기법입니다.

</br>

<div style="text-align:center">

</div>

## Luong Attention 구현
> 감성 분류에서 Attention은 LSTM의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 시점 출력</mark>을 활용합니다.</br></br>
> 마지막 hidden_state만 사용하는 대신, 각 토큰의 중요도를 가중합으로 계산해 더 풍부한 표현을 만듭니다.

<div style="text-align:center">

</div>

In [ ]:
# TODO 7: LuongAttention 클래스를 정의해봅시다.

class LuongAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

    def forward(self, query, encoder_outputs):
        # query: (batch, 1, hidden_dim) — 현재 시점의 hidden_state
        # encoder_outputs: (batch, seq_len, hidden_dim) — 모든 시점의 출력
        # TODO 7-1: Attention Score를 계산해봅시다. (행렬 곱)
        # TODO 7-2: Softmax로 Attention Weights를 구해봅시다.
        # TODO 7-3: 가중합으로 Context Vector를 계산해봅시다.
        pass

In [ ]:
# TODO 8: LuongAttention 인스턴스를 생성해봅시다. (hidden_dim=256)



In [ ]:
# TODO 9: 테스트 입력을 준비하고 LSTM으로 encoder_outputs를 생성해봅시다.

text = "정말 재미있는 영화였습니다"
encoded = tokenizer.encode(text)
ids = encoded.ids[:64] + [0] * max(0, 64 - len(encoded.ids))
token_tensor = torch.LongTensor([ids]).to(device)

# 임시 임베딩 + LSTM으로 encoder_outputs 생성
test_embedding = nn.Embedding(vocab_size, 256, padding_idx=0).to(device)
test_lstm = nn.LSTM(256, 256, batch_first=True).to(device)

embedded = test_embedding(token_tensor)
encoder_outputs, (hidden_state, _) = test_lstm(embedded)

# query = 마지막 시점의 hidden state
query = hidden_state[-1].unsqueeze(1)  # (batch, 1, hidden_dim)

print(f"encoder_outputs shape: {encoder_outputs.shape}")
print(f"query shape:           {query.shape}")

In [ ]:
# TODO 10: attention에 query와 encoder_outputs를 입력하여 결과를 확인해봅시다.



print(f"context_vector shape: {context_vector.shape}")
print(f"weights shape:        {weights.shape}")
print(f"\n각 토큰별 Attention 가중치 (학습 전 랜덤):")
for tok, w in zip(encoded.tokens, weights[0, 0, :len(encoded.tokens)]):

    print(f"  {tok:>10}: {w.item():.3f} {bar}")

💡torch.bmm (Batch Matrix Multiplication)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">bmm</mark>은 배치 단위 행렬 곱입니다. 일반 행렬 곱과 같지만 배치 차원을 유지합니다.</br></br>
> ```
> torch.bmm(A, B)
> A: (batch, n, m)  ×  B: (batch, m, k)  →  (batch, n, k)
> ```
> Attention에서의 사용:</br>
> ```
> Score:   bmm(query(b,1,256), encoder.T(b,256,seq)) → (b,1,seq)  각 시점 유사도
> Context: bmm(weights(b,1,seq), encoder(b,seq,256)) → (b,1,256)  가중합 벡터
> ```

💡Attention이 왜 효과적인가?
> 기존 분류기는 전체 입력을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">마지막 hidden_state 하나로 압축</mark>해야 했습니다.</br>
> Attention은 모든 시점의 출력을 활용해 중요한 단어에 더 집중하므로 정보 손실이 줄어듭니다.

💡Self-Attention과의 차이
> 여기서 배우는 Attention은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Cross-Attention</mark> (query ↔ encoder 출력 간)입니다.</br>
> Transformer의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Self-Attention</mark>은 같은 시퀀스 내부에서 관계를 계산합니다.

</br>

# LSTM+Attention 감성 분류기
> LSTM의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 시점 출력</mark>을 Attention으로 가중합하여 더 풍부한 문맥 벡터를 만듭니다.</br></br>
> 마지막 hidden_state와 context_vector를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">concat</mark>하여 분류합니다.

<div style="text-align:center">

</div>

In [ ]:
# TODO 11: AttentionClassifier 클래스를 정의해봅시다.

class AttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super().__init__()
        # TODO 11-1: 임베딩 레이어를 초기화해봅시다.
        # TODO 11-2: LSTM 레이어를 초기화해봅시다.
        # TODO 11-3: LuongAttention 모듈을 초기화해봅시다.
        # TODO 11-4: 드롭아웃 레이어를 초기화해봅시다.
        # TODO 11-5: FC 레이어를 초기화해봅시다. (hidden_dim*2 → num_classes)

    def forward(self, x):
        # TODO 11-6: 입력을 임베딩해봅시다.
        # TODO 11-7: LSTM에 통과시켜 모든 시점의 출력과 마지막 hidden_state를 얻어봅시다.
        # TODO 11-8: Attention으로 context_vector를 구해봅시다.
        # TODO 11-9: hidden_state와 context_vector를 concat하고 FC 레이어로 분류해봅시다.
        pass

attn_model = AttentionClassifier(vocab_size=vocab_size, embed_dim=256, hidden_dim=256).to(device)
print(f"파라미터 수: {sum(p.numel() for p in attn_model.parameters()):,}")

</br>

## LSTM+Attention 학습 및 평가

In [ ]:
# TODO 12: AttentionClassifier의 cross_entropy와 optimizer를 정의해봅시다.

cross_entropy_attn = nn.CrossEntropyLoss()
optimizer_attn = optim.Adam(attn_model.parameters(), lr=1e-3)

In [ ]:
# TODO 13: AttentionClassifier를 5 에폭 학습시켜봅시다.

attn_train_loss_list = []
attn_train_accuracy_list = []

for epoch in range(epochs):
    attn_model.train()
    running_loss = 0.0
    correct_count = 0
    total_count = 0

    for token_ids, labels in train_loader:
        token_ids = token_ids.to(device)
        labels    = labels.to(device)

        # TODO 13-1: 기울기를 초기화해봅시다.
        optimizer_attn.zero_grad()
        # TODO 13-2: 순전파를 수행해봅시다. (prediction과 attention_weights 반환)
        outputs, _ = attn_model(token_ids)
        # TODO 13-3: 손실을 계산해봅시다.
        loss = cross_entropy_attn(outputs, labels)
        # TODO 13-4: 역전파를 수행해봅시다.
        loss.backward()
        # TODO 13-5: 파라미터를 업데이트해봅시다.
        optimizer_attn.step()

        running_loss  += loss.item() * labels.size(0)
        _, predicted   = outputs.max(1)
        correct_count += (predicted == labels).sum().item()
        total_count   += labels.size(0)

    epoch_loss     = running_loss / total_count
    epoch_accuracy = correct_count / total_count
    attn_train_loss_list.append(epoch_loss)
    attn_train_accuracy_list.append(epoch_accuracy)
    print(f"Epoch {epoch+1}/{epochs}  loss: {epoch_loss:.4f}  accuracy: {epoch_accuracy:.4f}")

In [ ]:
# TODO 14: AttentionClassifier의 테스트 정확도를 측정해봅시다.

attn_model.eval()
correct_count = 0
total_count = 0

with torch.no_grad():
    for token_ids, labels in test_loader:
        token_ids = token_ids.to(device)
        labels    = labels.to(device)
        outputs, _ = attn_model(token_ids)
        _, predicted = outputs.max(1)
        correct_count += (predicted == labels).sum().item()
        total_count   += labels.size(0)

attn_accuracy = correct_count / total_count
print(f"LSTM+Attention 테스트 정확도: {attn_accuracy:.4f}  ({attn_accuracy*100:.2f}%)")

</br>

# 모델 비교: RNN vs LSTM vs LSTM+Attention
> 세 모델의 학습 곡선과 최종 테스트 정확도를 비교합니다.</br></br>
> LSTM이 RNN보다, LSTM+Attention이 LSTM보다 개선되었는지 확인해봅시다.

In [ ]:
# TODO 15: 학습 손실 곡선과 테스트 정확도 비교 그래프를 그려봅시다.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 학습 손실 곡선
axes[0].plot(range(1, epochs+1), lstm_train_loss_list, 'b-o', label='LSTM', linewidth=2)
axes[0].plot(range(1, epochs+1), attn_train_loss_list, 'r-s', label='LSTM+Attention', linewidth=2)
axes[0].set_title('학습 손실 곡선', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 테스트 정확도 비교
model_names = ['LSTM', 'LSTM+Attention']
accuracies  = [lstm_accuracy, attn_accuracy]
colors      = ['#2196F3', '#FF5722']
bars = axes[1].bar(model_names, accuracies, color=colors, width=0.5, edgecolor='white')
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('테스트 정확도 비교', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Accuracy')
for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== 최종 성능 비교 ===")
print(f"LSTM              테스트 정확도: {lstm_accuracy:.4f}")
print(f"LSTM+Attention    테스트 정확도: {attn_accuracy:.4f}")
print(f"개선 폭: {(attn_accuracy - lstm_accuracy)*100:+.2f}%p")

</br>

# Attention 시각화
> 모델이 감성 분류 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 토큰에 집중했는지</mark> 히트맵으로 확인합니다.

In [ ]:
# TODO 16: 긍정/부정 문장에 대한 Attention 가중치를 시각화해봅시다.

sentences = [
    "정말 너무 재미있고 감동적인 영화였습니다",
    "최악의 영화입니다 완전히 시간 낭비였어요",
]

attn_model.eval()
fig, axes = plt.subplots(len(sentences), 1, figsize=(14, 4 * len(sentences)))

for idx, sentence in enumerate(sentences):
    encoded = tokenizer.encode(sentence)
    ids = encoded.ids[:64] + [0] * max(0, 64 - len(encoded.ids))
    token_tensor = torch.LongTensor([ids]).to(device)

    with torch.no_grad():
        logits, attention_weights = attn_model(token_tensor)

    prediction = logits.argmax(-1).item()
    label_str = "긍정" if prediction == 1 else "부정"

    # 특수 토큰([CLS], [SEP]) 제외 — 의미 없는 토큰이 높은 가중치를 받는 현상 방지
    tokens = encoded.tokens
    special_tokens = {"[CLS]", "[SEP]", "[PAD]", "[UNK]"}
    mask = [i for i, t in enumerate(tokens) if t not in special_tokens]
    filtered_tokens = [tokens[i] for i in mask]
    raw_weights = attention_weights[0, 0, :len(tokens)].cpu()
    filtered_weights = raw_weights[mask].numpy()

    ax = axes[idx] if len(sentences) > 1 else axes
    im = ax.imshow(filtered_weights.reshape(1, -1), aspect='auto', cmap='YlOrRd', vmin=0)
    ax.set_xticks(range(len(filtered_tokens)))
    ax.set_xticklabels(filtered_tokens, rotation=45, ha='right', fontsize=10)
    ax.set_yticks([])
    ax.set_title(f'"{sentence}"  →  예측: {label_str}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.02)

plt.tight_layout()
plt.show()

💡Attention 시각화 해석
> 색이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">진할수록</mark> 해당 토큰에 더 높은 가중치가 부여된 것입니다.</br></br>
> 긍정 문장에서는 "재미있", "감동" 등에, 부정 문장에서는 "최악", "낭비" 등에 집중하면 Attention이 잘 작동하는 것입니다.</br></br>
> 학습이 부족하면 가중치가 고르게 분산되어 패턴이 보이지 않을 수 있습니다.